<a href="https://colab.research.google.com/github/lewinskie254/EPL/blob/main/EPL_model_development_collab_v.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [42]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import log_loss, accuracy_score

In [43]:
df = pd.read_csv("/features.csv")
df = df.drop(columns='Unnamed: 0')

In [44]:
train_end = int(0.7 * len(df))
val_end = int(0.85 * len(df))

train = df.iloc[:train_end]
val = df.iloc[train_end:val_end]
test = df.iloc[val_end:]

In [45]:
features = [col for col in df.columns if col.lower() != 'target']
target = 'Target'

In [46]:
tscv = TimeSeriesSplit(n_splits=5)

model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    device="cuda",
    predictor="gpu_predictor",
    random_state=42,
    eval_metric="mlogloss"
)

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
    "min_child_weight": [1, 3, 5]
}


In [ ]:
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring="neg_log_loss",
    cv=tscv,
    verbose=2,
    n_jobs=-1,
    return_train_score=True
)

grid.fit(train[features], train[target])

Fitting 5 folds for each of 729 candidates, totalling 3645 fits


In [ ]:
results = pd.DataFrame(grid.cv_results_)
results_sorted = results.sort_values("rank_test_score")

results_sorted[[
    "params",
    "mean_test_score",
    "std_test_score",
    "mean_train_score"
]].head(10)
results_sorted["log_loss"] = -results_sorted["mean_test_score"]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
model_path = "/content/drive/MyDrive/xgb_model.json"

In [ ]:
model = grid.best_estimator_
model.save_model(model_path)